# 01 – Thống kê mô tả (W1-07) và kiểm tra chất lượng dữ liệu (W1-08)

Hướng dẫn từng bước: `docs/huong_dan/W1-07.md`, `docs/huong_dan/W1-08.md`.
Ô có chữ **TODO** là phần bạn tự viết. Các ô khác chạy được ngay.
Hình vẽ lưu vào `results/figures/eda_*.png` để dán vào báo cáo mục 2.

In [ ]:
# Ô 0 – chuẩn bị môi trường Colab (chạy mỗi lần mở notebook)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'      # thư mục Drive chung có data/processed
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from src.data import DATASET_SPLITS, LABELS, load_split
from src.embeddings import tokenize

# Gộp mọi bộ / split vào 1 bảng để thống kê
df = pd.concat([load_split(d, s) for d, splits in DATASET_SPLITS.items() for s in splits], ignore_index=True)
df['tap'] = df['source'] + '_' + df['split']
df.groupby('tap').size()

## 1. Phân bố nhãn từng tập (W1-07)

In [ ]:
pd.crosstab(df['tap'], df['label']).reindex(columns=LABELS)

**TODO 1:** chuyển bảng trên thành tỉ lệ % theo hàng (gợi ý: `pd.crosstab(..., normalize='index')`) và viết 1–2 câu nhận xét:
bộ nào cân bằng, bộ nào lệch về nhãn nào.

In [ ]:
# TODO 1

## 2. Độ dài câu (số âm tiết)

In [ ]:
df['len_p'] = df['premise'].map(lambda s: len(tokenize(s)))
df['len_h'] = df['hypothesis'].map(lambda s: len(tokenize(s)))
df.groupby('tap')[['len_p', 'len_h']].agg(['mean', 'median', 'max']).round(1)

**TODO 2:** vẽ histogram độ dài hypothesis của 3 tập `xnli_train`, `vianli_train`, `vinli_test` trên cùng một hình
(gợi ý: vòng `for` + `plt.hist(..., bins=40, alpha=0.5, label=...)`), lưu `results/figures/eda_do_dai.png`.
Nhận xét: bộ nào câu dài hơn? Độ dài hypothesis có khác nhau giữa các nhãn không? (`df.groupby(['tap','label'])['len_h'].mean()`)

In [ ]:
# TODO 2

## 3. Độ trùng từ giữa premise và hypothesis (Jaccard)

In [ ]:
def jaccard(p, h):
    a, b = set(t.lower() for t in tokenize(p)), set(t.lower() for t in tokenize(h))
    return len(a & b) / max(1, len(a | b))

df['jaccard'] = [jaccard(p, h) for p, h in zip(df['premise'], df['hypothesis'])]
(df.pivot_table(index='label', columns='tap', values='jaccard') * 100).round(1).reindex(LABELS)

**TODO 3:** vẽ biểu đồ cột (nhãn × bộ dữ liệu) cho bảng trên, lưu `results/figures/eda_jaccard.png`.
Nhận xét: ở bộ nào entailment trùng từ nhiều nhất? Ở ViANLI thì sao? (So với `docs/ket_qua_m2.md` mục 4 và bài ViANLI Bảng 6.)

In [ ]:
# TODO 3

## 4. Từ phủ định theo nhãn

In [ ]:
NEG = r'\b(không|chưa|chẳng|chả|đâu có|không hề|chưa từng)\b'
# TODO 4: tạo cột df['h_neg'] = hypothesis có từ phủ định (True/False) bằng df['hypothesis'].str.contains(NEG, case=False)
#         rồi tính tỉ lệ % theo (tap, label): df.pivot_table(index='label', columns='tap', values='h_neg')
#         Câu hỏi: ở XNLI, từ phủ định gắn với nhãn nào? ViNLI và ViANLI có giống không?
#         (bài ViANLI mục 7.6: ở ViNLI "không" lại gắn với entailment)

---
# Kiểm tra chất lượng dữ liệu (W1-08)

In [ ]:
# 5. Câu rỗng hoặc quá ngắn
df[(df['len_p'] < 3) | (df['len_h'] < 2)][['tap', 'premise', 'hypothesis', 'label']].head(20)

In [ ]:
# 6. Cặp câu trùng lặp trong cùng một tập
dup = df[df.duplicated(['tap', 'premise', 'hypothesis'], keep=False)]
print(dup.groupby('tap').size())
# TODO 6: trong các cặp trùng, có cặp nào cùng premise + hypothesis nhưng KHÁC nhãn không? (gợi ý: groupby + nunique)

In [ ]:
# 7. Rò rỉ train -> test: cặp câu của test có xuất hiện trong train không?
def leak(train_tap, test_tap):
    tr = set(zip(df.loc[df.tap == train_tap, 'premise'], df.loc[df.tap == train_tap, 'hypothesis']))
    te = df[df.tap == test_tap]
    n = sum((p, h) in tr for p, h in zip(te['premise'], te['hypothesis']))
    return n, len(te)

for a, b in [('xnli_train', 'xnli_test'), ('vianli_train', 'vianli_test'), ('vianli_train', 'vinli_test')]:
    print(a, '->', b, leak(a, b))
# TODO 7: làm tương tự chỉ với premise (cùng premise, khác hypothesis) và ghi lại số lượng

**TODO 8 – tổng kết W1-08:** viết bảng *loại lỗi · số lượng · cách xử lý* (ví dụ: dấu cách trước dấu câu ở XNLI train ~91% → đã chuẩn hóa
trong `src/data.py`; id ViANLI chứa mã nhãn → không dùng id). Bảng này đưa vào báo cáo mục 2.